In [36]:
import pandas as pd
import nltk
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn

nltk.download("wordnet")
nltk.download("omw-1.4")            #for multilingual WordNet
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

## Preprocessing

In [37]:
dataset = pd.read_csv("DatasetDefinizioni.csv")

df = dataset.drop(columns=['Risposta numero'])

df.rename(columns={
    "(AG) Definizione del concetto ASTRATTO e GENERICO": "AG",
    "(AS) Definizione del concetto ASTRATTO e SPECIFICO": "AS",
    "(CG) Definizione del concetto CONCRETO e GENERICO": "CG",
    "(CS) Definizione del concetto CONCRETO e SPECIFICO": "CS",
    "(CM) Definizione del concetto misterioso (complesso)": "CM"
}, inplace=True)

df = df.apply(lambda col: col.str.replace(r'\xa0', ' ', regex=True)
                               .str.replace(r'\s+', ' ', regex=True)
                               .str.strip()
                               .str.lower())


## Synset Extraction Using Word Sense Disambiguation

This code performs semantic preprocessing of the definitions by converting each word into its corresponding WordNet synset representation. A helper function is defined to retrieve all synsets associated with the words in a given definition using WordNet.

Each definition is tokenized, and the Lesk algorithm is applied for **word sense disambiguation**, selecting the most appropriate synset for each word based on its context. Words for which no synset can be confidently assigned are ignored. The resulting synsets are collected for each definition and stored in a structured format per column.

In [ ]:
def get_synsets_from_definition(definition):
    # tokenize the input definition string into individual words and lowercase them 
    # to ensure consistency with the WordNet lexicon database
    words = word_tokenize(definition.lower())
    synsets = []

    for w in words:
        # fetch all synsets associated with the current word 'w'
        # .synsets(w) returns a list of synsets (synonym sets) for the given word
        # .extend() flattens the list by appending elements individually rather than adding a nested list
        synsets.extend(wn.synsets(w))

    return synsets

In [ ]:
data = {}

# find the minimum number of valid (non-NaN) definitions across all columns
# this ensures that all columns will have the exact same length in the final DataFrame
min_len = min(len(df[col].dropna()) for col in df.columns)

# word disambiguation
for column in df.columns:
    # Get the definitions for the current column, dropping NaN values and limiting to min_len
    definitions = df[column].dropna().tolist()[:min_len]

    column_synsets = []

    for definition in definitions:
        # Tokenize the definition into words and convert to lowercase
        tokens = word_tokenize(definition.lower())


        synsets_per_definition = [
            lesk(tokens, word).name()           # Get the synset name (e.g., 'dog.n.01') instead of the full synset object
            for word in tokens                  # Iterate over each word in the tokenized definition
            if lesk(tokens, word) is not None   # Ensure that lesk returns a valid synset before trying to access its name
        ]

        column_synsets.append(synsets_per_definition)

    data[column] = column_synsets

df_synsets = pd.DataFrame(data)
df_synsets.to_csv("DatasetDefinizioni_Synsets.csv", index=False)


df_synsets = df_synsets.drop(columns=['CM'])
df_synsets


,AG,AS,CG,CS
0,"[art.n.03, composed.a.01, tuneful.a.01, sound....","[philosophic.a.01, think.v.03, aim.v.01, astat...","[angstrom.n.01, plant.v.05, root.n.01, trunk.n...","[kitchen.n.01, tool.n.01, use.v.04, steep.v.02..."
1,"[angstrom.n.01, set.n.01, sound.v.04, make.v.1...","[arm.n.02, philosophy.n.02, focus.n.01, on.a.0...","[plant.n.01, different.a.01, type.v.01, generi...","[metallic_element.n.01, container.n.01, normal..."
2,"[group.n.02, every.s.01, musical.n.01, note.n....","[common.s.03, sentiment.n.01, accord.v.02, eve...","[high.a.04, solid.n.02, kind.n.01, plant.n.02,...","[small.n.02, container.n.01, liquid.n.02, use...."
3,"[phone.n.02, make.v.03, by.r.01, one.s.01, ore...","[doctrine.n.01, oregon.n.01, science.n.01, con...","[animation.n.01, organism.n.01, life.n.03, ind...","[creature.n.03, use.v.01, make.v.03, tea.n.05]"
4,"[deoxyadenosine_monophosphate.n.01, mixture.n....","[philosophy.n.02, randomness.n.01, branch.n.02...","[plant.n.02, construct.v.01, by.r.01, deoxyade...","[recipient.n.01, hour_angle.n.02, deoxyadenosi..."
5,"[associate_in_nursing.n.01, organization.n.04,...","[philosophy.n.02, study.n.02, be.v.01, right.n...","[deoxyadenosine_monophosphate.n.01, life.n.02,...","[angstrom.n.01, container.n.01, can.n.02, be.v..."
6,"[succession.n.03, fathom.v.02, aim.n.03, trip....","[branch.n.01, doctrine.n.01, purpose.n.01, ask...","[woody.a.01, plant.n.01, ramify.v.02, leaf.n.01]","[container.n.01, frequently.r.01, make.v.14, c..."
7,"[succession.n.03, sound.n.02, capable.a.03, ar...","[moral.a.01, principle.n.01, memorize.v.01, by...","[type.n.01, plant.n.02, composed.a.01, root.n....","[container.n.01, normally.r.01, cook.v.02, cer..."
8,"[set.n.01, sound.n.02, harmony.n.02, are.n.01,...","[set.n.01, conventionally.r.01, partake.v.02, ...","[organic.n.01, component.n.03, bear.v.11, deox...","[container.n.01, utilitarian.s.01, retain.v.03..."
9,"[danger.n.01, be.v.01, deoxyadenosine_monophos...","[deoxyadenosine_monophosphate.n.01, heuristic....","[angstrom.n.01, creature.n.03, be.v.01, associ...","[angstrom.n.01, microscope.n.01, be.v.01, angs..."


## Closest Synset Matching Using Semantic Similarity

For each definition, the WordNet synset most semantically similar to a predefined target concept is identified. A helper function is defined to compare a list of synset names with a target synset using the **Wu–Palmer similarity measure (wup_similarity)**, which quantifies semantic relatedness based on the depth of concepts in the WordNet hierarchy.

For each candidate synset, the function computes its similarity to the target synset and keeps track of the one with the highest score. The final output for each definition is the synset with the strongest semantic correspondence.

A dictionary of expected target concepts is defined for each column (e.g., *music*, *ethics*, *tree*, *teapot*), and their corresponding primary WordNet synsets are extracted. Then, for each definition in the dataset, the closest synset to the target concept is computed and stored.

In [ ]:
def closest_synset_to_target(synset_names, target_synset):
    best_name = None
    best_score = -1

    for name in synset_names:
        try:
            # reconstruct the WordNet Synset object using its string identifier (e.g., 'dog.n.01')
            syn = wn.synset(name)
            # compute the similarity score between the current synset and the target
            # use '0' as a safety fallback in case wup_similarity returns None
            score = syn.wup_similarity(target_synset) or 0
            if score > best_score:
                # update the optimal match if the current similarity score is higher
                best_score = score
                best_name = name
        except Exception:
            continue

    return best_name, f"{best_score * 100:.2f}%"


In [ ]:
word_expected = {
    "AG": "music",
    "AS": "ethics",
    "CG": "tree",
    "CS": "teapot",
}

target_synsets = {}
# retrieved the synsets of the expected words, and the first one is taken as the target synset for each column
for col, word in word_expected.items(): # .items() breaks the dictionary into (key, value) pairs, unpacking them into 'col' and 'word'
    syns = wn.synsets(word)
    if syns:
        target_synsets[col] = syns[0]

# dictionary to hold the final similarity mapping results
closest_data = {}

for col in df_synsets.columns:
    # if the current column does not have a corresponding target synset, skip it
    if col not in target_synsets:
        continue

    target_syn = target_synsets[col]

    # for each list of synset names in the current column, find the closest synset to the target synset and store the results
    closest_data[col] = [
        closest_synset_to_target(synset_list, target_syn)
        for synset_list in df_synsets[col]
    ]

df_closest = pd.DataFrame(closest_data)
df_closest

,AG,AS,CG,CS
0,"(vocal_music.n.01, 90.91%)","(ideal.n.01, 50.00%)","(root.n.01, 44.44%)","(tool.n.01, 70.00%)"
1,"(citizenry.n.01, 44.44%)","(behavior.n.01, 50.00%)","(plant.n.01, 44.44%)","(container.n.01, 82.35%)"
2,"(note.n.03, 46.15%)","(life.n.02, 50.00%)","(plant.n.02, 82.35%)","(container.n.01, 82.35%)"
3,"(phone.n.02, 36.36%)","(doctrine.n.01, 50.00%)","(flower.n.01, 76.19%)","(creature.n.03, 42.11%)"
4,"(lyric.n.01, 46.15%)","(politics.n.05, 50.00%)","(plant.n.02, 82.35%)","(hat.n.01, 52.63%)"
5,"(voice.n.02, 66.67%)","(study.n.02, 46.15%)","(organism.n.01, 75.00%)","(container.n.01, 82.35%)"
6,"(emotion.n.01, 36.36%)","(doctrine.n.01, 50.00%)","(plant.n.01, 44.44%)","(container.n.01, 82.35%)"
7,"(emotion.n.01, 36.36%)","(principle.n.01, 46.15%)","(plant.n.02, 82.35%)","(container.n.01, 82.35%)"
8,"(harmony.n.02, 90.91%)","(set.n.01, 40.00%)","(leaf.n.01, 44.44%)","(container.n.01, 82.35%)"
9,"(danger.n.01, 36.36%)","(aid.n.02, 50.00%)","(creature.n.03, 63.16%)","(microscope.n.01, 57.14%)"


## Word Sense Disambiguation Using Lesk Algorithm

For each row in the dataset, Word Sense Disambiguation is performed using the **Lesk algorithm**, with the goal of selecting the most appropriate WordNet synset for a predefined target word based on its contextual definition. A dictionary of expected target concepts is defined for each column (e.g., *music*, *ethics*, *tree*, *teapot*).

The algorithm iterates over each definition in the DataFrame row by row. For each concept column, the corresponding definition is tokenized and used as contextual information to disambiguate the target word via the Lesk algorithm. This method selects the synset whose definition has the highest overlap with the context words, thus approximating the intended meaning of the word in that specific usage.

If a synset is successfully identified, its name is stored; otherwise, a missing value is recorded.

In [ ]:
# List of dictionaries to hold the results for each row
records = []  

# Expected words for each column
word_expected = {
    "AG": "music",
    "AS": "ethics",
    "CG": "tree",
    "CS": "teapot",
}

# Iterate over each row in the DataFrame
for i in range(len(df)):
    # Hold the synset results for the current row
    row = {}

    # Iterate over each column and disambiguate the target word using the definition as context
    for column in word_expected:
        definition = df[column].iloc[i]
        # disambiguate the target word using the current definition as context
        # the Lesk algorithm looks up all possible meanings (synsets) of the target 'word_expected' 
        # in the WordNet dictionary and selects the one that shares the highest word overlap 
        # with the definition text ('context_sentence').
        synset = lesk(word_tokenize(definition), word_expected[column])

        # if Lesk successfully finds a sense match, extract its unique string name;
        # otherwise, assign None to represent an unresolved ambiguity or missing data.
        row[column] = synset.name() if synset else None

    records.append(row)

df_results = pd.DataFrame(records)
df_results

,AG,AS,CG,CS
0,music.n.04,ethical_motive.n.01,tree.n.01,teapot.n.01
1,music.n.04,ethical_motive.n.01,tree.n.01,teapot.n.01
2,music.n.01,ethical_motive.n.01,tree.n.01,teapot.n.01
3,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
4,music.n.01,ethic.n.01,tree.n.01,teapot.n.01
5,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
6,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
7,music.n.04,ethic.n.01,tree.n.01,teapot.n.01
8,music.n.01,ethics.n.02,tree.n.01,teapot.n.01
9,music.n.01,ethic.n.01,corner.v.02,teapot.n.01
